In [5]:
import sys
sys.path.append('..')

import os
os.environ["KERAS_BACKEND"] = "torch"

import time
import keras
import numpy as np
# Necesario para TextVectorization y tf.data.
import tensorflow as tf
from models.training import compile_model, get_callbacks
from config.settings import Settings
from features.embeddings import load_gensim_embeddings
from datasets.dataset import create_dataset
from features.vectorizer import build_vectorizer
from datasets.loader import load_splits, save_json
from models.siamese_lstm import SiameseLSTM
from datasets.paths import ProjectPaths


In [6]:
print(keras.config.backend())


torch


In [7]:
SEED = 42
np.random.seed(SEED)


In [8]:
settings = Settings()
print(settings)


embed_dim=400 batch_size=64 mlp_dropout=0.4 lstm_dropout=0.3 pooling='mean' similarity='mlp' hidden_dim=64 bidirectional=True mlp_layers=[32] concat_features=['diff'] epochs=20 augmented_data=False siamese_name='bilstm_mean_mlp_noaug'


In [9]:
paths = ProjectPaths(siamese_name=settings.siamese_name)


In [10]:
if settings.augmented_data:
	max_len = 27
	train_dir = paths.augmented_dir
else:
	max_len = 26
	train_dir = paths.processed_dir


In [11]:
splits = {
	"train": train_dir,
	"dev": paths.processed_dir,
    "test": paths.processed_dir
}

datasets = load_splits(splits)

train_df = datasets["train"]
dev_df = datasets["dev"]
test_df = datasets["test"]


In [12]:
print("Train length:", len(train_df))
print("Dev length:", len(dev_df))


Train length: 5741
Dev length: 1497


In [13]:
all_sentences = list(train_df["sentence1"]) + list(train_df["sentence2"])

vectorizer = build_vectorizer(all_sentences, max_len)

vocab = vectorizer.get_vocabulary()
word2idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocabulary size: {len(vocab)}")

vectorizer_model = keras.Sequential([vectorizer])
vectorizer_model.save(paths.vectorizer_path)


Vocabulary size: 13698


c:\Users\malos\Documents\GitHub\JustShare\server\.venv\Lib\site-packages\keras\src\saving\saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


In [14]:
embedding_matrix = load_gensim_embeddings(paths.word2vec_path, word2idx, settings.embed_dim)

np.save(paths.embedding_path, embedding_matrix)


Found 13300/13698 words


In [15]:
train_dataset = create_dataset(train_df, vectorizer, settings.batch_size, shuffle=True)
dev_dataset = create_dataset(dev_df, vectorizer, settings.batch_size)


In [16]:
for (sent1, sent2), y in train_dataset.take(1):
	print("sent1:", sent1.shape)
	print("sent2:", sent2.shape)
	print("y:", y.shape)


sent1: (64, 26)
sent2: (64, 26)
y: (64,)


In [17]:
model = SiameseLSTM(
	vocab_size=len(vocab),
	embedding_dim=settings.embed_dim,
	hidden_dim=settings.hidden_dim,
	mlp_dropout=settings.mlp_dropout,
	lstm_dropout=settings.lstm_dropout,
	embedding_matrix=embedding_matrix,
	pooling=settings.pooling,
	similarity=settings.similarity,
	mlp_layers=settings.mlp_layers,
	bidirectional=settings.bidirectional,
	concat_features=settings.concat_features,
    name=settings.siamese_name
)


In [18]:
if model.mlp:
	model.mlp.summary()


Model: "mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,161 (16.25 KB)

 Trainable params: 4,161 (16.25 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
head_model = model.get_head_model()
head_model.summary()


Model: "siamese_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 400) │  5,479,200 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast (Cast)         │ (None, None)      │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ (None, None, 128) │    238,080 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, None, 1)   │          0 │ cast[0][0]        │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, None, 128) │          0 │ bilstm[0][0],     │
│                     │                   │            │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum_1 (Sum)         │ (None, 1)         │          0 │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum (Sum)           │ (None, 128)       │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1)         │          0 │ sum_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ true_divide         │ (None, 128)       │          0 │ sum[0][0],        │
│ (TrueDivide)        │                   │            │ add[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,717,280 (21.81 MB)

 Trainable params: 238,080 (930.00 KB)

 Non-trainable params: 5,479,200 (20.90 MB)

In [20]:
dummy_sent1 = tf.zeros((1, max_len), dtype=tf.int32)
dummy_sent2 = tf.zeros((1, max_len), dtype=tf.int32)

model((dummy_sent1, dummy_sent2))

model.summary()


Model: "bilstm_mean_mlp_noaug"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 400)      │     5,479,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, None, 128)      │       238,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mlp (Sequential)                │ (None, 1)              │         4,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,721,441 (21.83 MB)

 Trainable params: 242,241 (946.25 KB)

 Non-trainable params: 5,479,200 (20.90 MB)

In [21]:
model = compile_model(model)

callbacks = get_callbacks(paths.siamese_path)


In [22]:
start_time = time.perf_counter()

history = model.fit(
	train_dataset,
	validation_data=dev_dataset,
	epochs=settings.epochs,
	callbacks=callbacks
)

train_time = time.perf_counter() - start_time

np.save(paths.history_path, history.history)


Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - loss: 0.0847 - mae: 0.2487 - rmse: 0.2910 - val_loss: 0.0901 - val_mae: 0.2536 - val_rmse: 0.3002 - learning_rate: 0.0010
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - loss: 0.0767 - mae: 0.2346 - rmse: 0.2769 - val_loss: 0.0794 - val_mae: 0.2378 - val_rmse: 0.2817 - learning_rate: 0.0010
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - loss: 0.0700 - mae: 0.2218 - rmse: 0.2646 - val_loss: 0.0730 - val_mae: 0.2241 - val_rmse: 0.2701 - learning_rate: 0.0010
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - loss: 0.0642 - mae: 0.2115 - rmse: 0.2534 - val_loss: 0.0681 - val_mae: 0.2150 - val_rmse: 0.2609 - learning_rate: 0.0010
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - loss: 0.0578 - mae: 0.1980 - rmse: 0.2404 - val_loss: 0.0658 - val_mae: 0.2091 - val_rmse: 0.2565 - learning_rate: 0.0010
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - loss: 0.0541 - mae: 0.1903 - rmse: 0.2327 - val_loss: 0.0621 - val_mae: 0

In [23]:
run_config = {
    "sequence_length": max_len,
    "data_augmentation": settings.augmented_data,
    "train_time_s": train_time
}

save_json(run_config, paths.config_path)
